This has been run :)

but I noticed a bug with the draw tirgger count???

TODO: Investigate and fix that?

Imports

Constants

In [10]:
import db_operations
import helpers

import pandas as pd
from pymongo import UpdateOne


STAND_HEALS = [
    "Stealth Fiend, Amaviera",
    "Incorruptible Holy Light, Eufha",
    "Lady Healer of the Creaking World",
    "Invigorate Sage",
    "Zypsophila Fairy, Asher",
    "Two Of Us, Rhyme",
]
CRIT_HEALS = [
    'Cure Flare Dracokid',
    'Brilliant Floral, Uania',
    'Whistling Arrow of Recursion, Obifold',
    'Heartiness Tear Sorceress',
    'Alchemic Hedgehog',
    'Two Of Us, Flow',
]

DRAWS = [
    'Flare Veil Dragon',
    'Rouse Wildmaster, Riley',
    'Ameliorate Connector',
    'Protection Magic, Prorobi',
    'Serene Maiden, Lena',
    'Transparent Snowy Night, Beretoi',
]

REGALIS = [
    "Primordial Heartblaze",
    "Shroud in Darkness",
    "Sound of Wave",
    "Fire Regalis",
    "Union the Sky",
    "Protection: Twincast",
    "Radiance Caliburn",
    "Bracing Angel Ladder",
    "Forbidoll Surrogate",  "'Mysterious Joker' Joker",
    "Evergreen Transhpere", "'Homeroom-Ninja Mr. Shinobu' Mr. Shinobu",
    "Gratias Gradale",      "'Croket!' Croket",
]


REGALIS_MAP = {
    "Primordial Heartblaze": None,
    "Shroud in Darkness": None,
    "Sound of Wave": None,
    "Fire Regalis": None,
    "Union the Sky": None,
    "Protection: Twincast": None,
    "Radiance Caliburn": None,
    "Bracing Angel Ladder": None,
    "Forbidoll Surrogate": None,  
    "'Mysterious Joker' Joker": "Forbidoll Surrogate",
    "Evergreen Transhpere": None, 
    "'Homeroom-Ninja Mr. Shinobu' Mr. Shinobu": "Evergreen Transhpere",
    "Gratias Gradale": None,      
    "'Croket!' Croket": "Gratias Gradale",
}

def extract_regalis(decks_dict, regalis_list):
    for card_name in regalis_list:
        if helpers.card_count_in_deck(decks_dict, card_name):
            if REGALIS_MAP[card_name] != None:
                return REGALIS_MAP[card_name]
            else:
                return card_name
        
    return "None"

Helper Functions

Get Data

In [ ]:
# full_df = db_operations.get_table('main_table', 'all_events')
# full_df = db_operations.get_answer_from_table('main_table', 'all_events', {
#     'location': 'Greece'
# })

In [8]:

full_df.sort_values('rank', inplace=True)
full_df.head()

,_id,rank,name,boss,wins,nation,decklog,deck,location,region,date
0,69b50404501e4375b37a5b24,1,Champion,"DZ-BT12/SR32EN : Deeplands, Reguregnus",8,Stoicheia,6RWS3,"{'RideDeck': {'DZ-BT12/SR32EN : Deeplands, Reg...",Greece,EU,"March 7, 2026"
28,69b50404501e4375b37a5b25,2,Champion,"DZ-BT02/FR05EN : Rewrite the Star, Vyrgilla",7,Dragon Empire,2AU59,{'RideDeck': {'DZ-BT02/FR05EN : Rewrite the St...,Greece,EU,"March 7, 2026"
37,69b50404501e4375b37a5b26,3,Champion,"DZ-SS09/001EN : Rewrite the Star, Vyrgilla",6,Dragon Empire,4MKFH,{'RideDeck': {'DZ-SS09/001EN : Rewrite the Sta...,Greece,EU,"March 7, 2026"
1,69b50404501e4375b37a5b27,4,Asta,"DZ-BT12/014EN : Deeplands, Reguregnus",5,Stoicheia,5EZW7,"{'RideDeck': {'DZ-BT12/014EN : Deeplands, Regu...",Greece,EU,"March 7, 2026"
13,69b50404501e4375b37a5b28,5,Kapsoulas,"DZ-BT11/016EN : Fated One of Ever-changing, Kr...",5,Lyrical Monasterio,5EG5R,{'RideDeck': {'DZ-BT11/016EN : Fated One of Ev...,Greece,EU,"March 7, 2026"


Feature Engineering

In [11]:
full_df.loc[:,'stand_heal_count'] = full_df.deck.apply(lambda deck: helpers.card_type_in_deck(deck, STAND_HEALS))

full_df.loc[:,'crit_heal_count']  = full_df.deck.apply(lambda deck: helpers.card_type_in_deck(deck, CRIT_HEALS))

full_df.loc[:,'draw_count']       = full_df.deck.apply(lambda deck: helpers.card_type_in_deck(deck, REGALIS))

full_df.loc[:, 'regalis_piece']   = full_df.deck.apply(lambda deck: extract_regalis(deck, REGALIS))

full_df.loc[:, 'boss'] = full_df.boss.apply(lambda boss: helpers.extract_card_name(boss))

# full_df.sort_values('rank', inplace=True)
full_df.head()

,_id,rank,name,boss,wins,nation,decklog,deck,location,region,date,stand_heal_count,crit_heal_count,draw_count,regalis_piece
0,69b50404501e4375b37a5b24,1,Champion,"Deeplands, Reguregnus",8,Stoicheia,6RWS3,"{'RideDeck': {'DZ-BT12/SR32EN : Deeplands, Reg...",Greece,EU,"March 7, 2026",0,0,1,Protection: Twincast
28,69b50404501e4375b37a5b25,2,Champion,"Rewrite the Star, Vyrgilla",7,Dragon Empire,2AU59,{'RideDeck': {'DZ-BT02/FR05EN : Rewrite the St...,Greece,EU,"March 7, 2026",1,0,1,Union the Sky
37,69b50404501e4375b37a5b26,3,Champion,"Rewrite the Star, Vyrgilla",6,Dragon Empire,4MKFH,{'RideDeck': {'DZ-SS09/001EN : Rewrite the Sta...,Greece,EU,"March 7, 2026",2,0,1,Protection: Twincast
1,69b50404501e4375b37a5b27,4,Asta,"Deeplands, Reguregnus",5,Stoicheia,5EZW7,"{'RideDeck': {'DZ-BT12/014EN : Deeplands, Regu...",Greece,EU,"March 7, 2026",2,0,1,Fire Regalis
13,69b50404501e4375b37a5b28,5,Kapsoulas,"Fated One of Ever-changing, Krysrain Cadenza",5,Lyrical Monasterio,5EG5R,{'RideDeck': {'DZ-BT11/016EN : Fated One of Ev...,Greece,EU,"March 7, 2026",0,0,1,Protection: Twincast


Save

In [12]:

updates = []
for i, row in full_df.iterrows():
    filter = {'_id': row['_id']}
    update = {
        '$set': {
            'boss': row['boss'],
            'stand_heal_count': row['stand_heal_count'],
            'crit_heal_count': row['crit_heal_count'],
            'draw_count': row['draw_count'],
            'regalis_piece': row['regalis_piece'],
        }
    }
    updates.append(UpdateOne(filter, update))

db_operations.update_table('main_table', 'all_events',  updates)